# QA Agent Web UI (Notebook)

This notebook starts a Gradio Web UI on top of the existing CLI codebase.

Run order:
1. Run the initialization code cell below
2. Run the last cell to launch the UI
3. Open the local link shown in output (usually `http://127.0.0.1:7860`)

In [1]:
from pathlib import Path
import sys

# Ensure the notebook can import modules from the project root
project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

try:
    import gradio as gr
except ImportError:
    %pip install -q gradio
    import gradio as gr

from main import initialize_agent

agent = initialize_agent()
print("QA Agent initialized for Web UI")

[2026-03-17 21:41:28,064] [main] [INFO] Initializing QA Agent components...
[2026-03-17 21:41:28,212] [src.retrievers.hybrid_retriever] [INFO] Loaded 2067 document chunks
[2026-03-17 21:41:28,310] [src.retrievers.hybrid_retriever] [INFO] Knowledge Graph enabled
[2026-03-17 21:41:28,466] [src.retrievers.hybrid_retriever] [INFO] HybridRetriever initialized with 2067 documents
[2026-03-17 21:41:28,894] [main] [INFO] QA Agent initialized successfully


QA Agent initialized for Web UI


In [2]:
def _format_details(response):
    breakdown = response.confidence_breakdown or {}
    breakdown_text = (
        f"- Retrieval Quality: {breakdown.get('retrieval_quality', 0.0):.1%}\n"
        f"- Answer Consistency: {breakdown.get('answer_consistency', 0.0):.1%}\n"
        f"- Citation Coverage: {breakdown.get('citation_coverage', 0.0):.1%}\n"
        f"- Final Weighted Score: {breakdown.get('final', response.confidence_score):.1%}"
    )
    sources = "\n".join(f"- {src}" for src in sorted(set(response.sources))) if response.sources else "- (none)"
    citations = "\n".join(f"- {c}" for c in sorted(set(response.legal_citations))) if response.legal_citations else "- (none)"
    return (
        f"**Confidence**: {response.confidence_score:.1%}\n{breakdown_text}\n\n"
        f"**Processing time**: {response.processing_time:.2f}s\n\n"
        f"**Sources**\n{sources}\n\n"
        f"**Legal citations**\n{citations}"
    )


def _format_conversation_history():
    history = agent.get_conversation_history()
    if not history:
        return "No conversation history yet."

    lines = []
    for idx, item in enumerate(reversed(history), 1):
        response = item.get("response", {})
        timestamp = item.get("timestamp")
        timestamp_text = timestamp.strftime("%Y-%m-%d %H:%M:%S") if hasattr(timestamp, "strftime") else str(timestamp)
        query = item.get("query", "")
        answer = response.get("answer", "")
        confidence = response.get("confidence_score", 0.0)
        lines.append(
            f"### {idx}. {query}\n"
            f"- Timestamp: {timestamp_text}\n"
            f"- Confidence: {confidence:.1%}\n"
            f"- Answer preview: {answer[:240]}{'...' if len(answer) > 240 else ''}"
        )

    return "\n\n".join(lines)


def show_history():
    return _format_conversation_history()


def ask_agent(user_query, history):
    query = (user_query or "").strip()
    if not query:
        return "", history, "Please enter a question.", _format_conversation_history()

    history = history or []

    try:
        response = agent.process_query(query)
        details = _format_details(response)
        history = history + [
            {"role": "user", "content": query},
            {"role": "assistant", "content": response.answer}
        ]
        return "", history, details, _format_conversation_history()
    except Exception as exc:
        error_message = str(exc)
        history = history + [
            {"role": "user", "content": query},
            {"role": "assistant", "content": "I encountered a temporary processing error. Please try again in a few seconds."}
        ]
        details = (
            "**Error**: Request failed.\n\n"
            f"**Technical detail**: {error_message}\n\n"
            "If this is a network SSL/proxy issue, retry the same query or verify your network/proxy settings."
        )
        return "", history, details, _format_conversation_history()


def clear_chat():
    agent.clear_history()
    return [], "Conversation history has been cleared.", "No conversation history yet."


with gr.Blocks(title="Singapore Legal & Tax QA Agent") as demo:
    gr.Markdown("## Singapore Legal & Tax QA Agent")
    gr.Markdown("Ask Singapore legal and tax questions, get confidence-scored answers with citations")

    chatbot = gr.Chatbot(height=450, label="Conversation")
    details = gr.Markdown("Query metadata will be shown here.")

    with gr.Row():
        query_input = gr.Textbox(label="Your Question", placeholder="Ask about Singapore legal/tax rules...", scale=5)
        send_btn = gr.Button("Send", variant="primary", scale=1)

    with gr.Accordion("Conversation History", open=False):
        history_panel = gr.Markdown("No conversation history yet.")

    with gr.Row():
        clear_btn = gr.Button("Clear History")
        show_history_btn = gr.Button("Show Conversation History")

    send_btn.click(ask_agent, inputs=[query_input, chatbot], outputs=[query_input, chatbot, details, history_panel])
    query_input.submit(ask_agent, inputs=[query_input, chatbot], outputs=[query_input, chatbot, details, history_panel])
    clear_btn.click(clear_chat, outputs=[chatbot, details, history_panel])
    show_history_btn.click(show_history, outputs=[history_panel])

demo.launch(inline=True, share=False)

[2026-03-17 21:41:29,033] [httpx] [INFO] HTTP Request: GET http://127.0.0.1:7860/gradio_api/startup-events "HTTP/1.1 200 OK"
[2026-03-17 21:41:29,050] [httpx] [INFO] HTTP Request: HEAD http://127.0.0.1:7860/ "HTTP/1.1 200 OK"


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


[2026-03-17 21:41:30,116] [httpx] [INFO] HTTP Request: GET https://api.gradio.app/pkg-version "HTTP/1.1 200 OK"


In [ ]:
# Stop the running Gradio app in this notebook session.
STOP_NOW = False

if STOP_NOW:
    stopped = False
    try:
        gr.close_all()
        stopped = True
    except Exception as exc:
        print(f"gr.close_all() not available or failed: {exc}")

    try:
        demo.close()
        stopped = True
    except Exception as exc:
        print(f"demo.close() failed or demo not initialized: {exc}")

    if stopped:
        print("Gradio service stopped.")
    else:
        print("No active Gradio service found.")

[2026-03-17 21:41:59,704] [src.agents.qa_agent] [INFO] Processing query: what is the tax rate for the individual?...
[2026-03-17 21:41:59,705] [src.agents.qa_agent] [INFO] Query type: tax
[2026-03-17 21:41:59,705] [src.retrievers.hybrid_retriever] [INFO] retrieve() enter | docs=2067 | enable_rerank=True | has_bm25=True
[2026-03-17 21:41:59,712] [src.retrievers.hybrid_retriever] [INFO] Reranking top 20 candidates with Cohere reranker
[2026-03-17 21:42:00,483] [httpx] [INFO] HTTP Request: POST https://api.cohere.com/v1/rerank "HTTP/1.1 200 OK"
[2026-03-17 21:42:00,487] [src.retrievers.hybrid_retriever] [INFO] === RERANK DEBUG START ===
[2026-03-17 21:42:00,488] [src.retrievers.hybrid_retriever] [INFO] Query: what is the tax rate for the individual?
[2026-03-17 21:42:00,493] [src.retrievers.hybrid_retriever] [INFO] Rank change: 2 -> 1 | Income Tax Act 1947 >> PART 11 RATES OF TAX >> 43. Rate of tax upon companies and others: (i) that takes place in Singap...
[2026-03-17 21:42:00,494] [src

: 